In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# =============================================================================
# Cell 1: Setup and Configuration
# =============================================================================

import os
import gc
import json
import gzip
import logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict

import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.csv as pacsv
from tqdm.auto import tqdm

# Configure logging (Professional, no symbols)
logging.basicConfig(
    level=logging.INFO,
    force = True,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
log = logging.getLogger("DataSplitPipeline")

# Polars configuration for large datasets
pl.Config.set_tbl_rows(10)
pl.Config.set_fmt_str_lengths(50)
os.environ["OPENBLAS_NUM_THREADS"] = "1"

@dataclass
class SplitConfig:
    """Configuration for Data Splitting Pipeline."""

    # Core Ratios and Thresholds
    k_user: int = 15
    k_book: int = 10
    train_ratio: float = 0.8
    valid_ratio: float = 0.1

    # Rating Definitions
    explicit_min_rating: int = 1
    explicit_max_rating: int = 5
    positive_min_rating: int = 4
    negative_max_rating: int = 3

    # Base Directories
    base_dir: Path = field(default_factory=lambda: Path("/content/drive/My Drive/Thesis/book_recsys"))
    thesis_dir: Path = field(default_factory=lambda: Path("/content/drive/My Drive/Thesis/"))
    # Pipeline Control Flags
    skip_stage1: bool = False
    skip_stage2: bool = False
    skip_stage3: bool = False

    def __post_init__(self) -> None:
        """Resolve paths based on the base directory."""
        self.data_dir = self.base_dir / "data"
        self.raw_dir = self.thesis_dir / "Data"

        # Updated output directory per requirements
        self.output_dir = self.data_dir / "processed_3"
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # Input Files
        self.raw_json_gz = self.raw_dir / "goodreads_interactions_dedup.json.gz"
        self.ingested_parquet = self.data_dir / "interactions_raw.parquet"
        self.kcore_parquet = self.data_dir / "interactions_kcore.parquet"

        # Output Files Configuration
        self.outputs: Dict[str, Path] = {
            "train_main": self.output_dir / "train_main.csv",
            "explicit_train": self.output_dir / "explicit_train.csv",
            "train_implicit": self.output_dir / "train_implicit.csv",
            "valid_pos": self.output_dir / "valid_pos.csv",
            "test_pos": self.output_dir / "test_pos.csv",
            "test_neg": self.output_dir / "test_neg.csv",
            "test_implicit": self.output_dir / "test_implicit.csv",
            "explicit_test": self.output_dir / "explicit_test.csv",
        }

# Verify configuration initialization
if __name__ == "__main__":
    cfg = SplitConfig()
    log.info("Configuration initialized successfully.")
    log.info(f"Target output directory: {cfg.output_dir}")

2026-04-18 10:58:57 [INFO] DataSplitPipeline: Configuration initialized successfully.
2026-04-18 10:58:57 [INFO] DataSplitPipeline: Target output directory: /content/drive/My Drive/Thesis/book_recsys/data/processed_3


In [ ]:
# =============================================================================
# Cell 2: Stage 1 - JSON to Parquet Ingestion (Memory Safe)
# =============================================================================

import gc
import json
import gzip
import logging
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

log = logging.getLogger("DataSplitPipeline")

def stage_1_ingestion(cfg) -> None:
    """
    Reads a massive JSON.gz file line-by-line, extracts only required fields,
    and streams to a Parquet file in chunks to prevent Out-Of-Memory (OOM) errors.
    """
    if cfg.skip_stage1:
        log.info("Stage 1 skipped via configuration.")
        return

    log.info("Stage 1: JSON.gz to Parquet Ingestion (Memory Safe Mode)")
    input_path = cfg.raw_json_gz
    output_path = cfg.ingested_parquet

    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found at: {input_path}")

    # Enforce strict schema to optimize PyArrow memory allocation
    schema = pa.schema([
        ('user_id', pa.string()),
        ('book_id', pa.string()),
        ('rating', pa.int16()),
        ('date_updated', pa.string())
    ])

    chunk_size = 100_000
    buffer = []
    writer = None

    log.info("Starting streaming process...")

    with gzip.open(input_path, 'rt', encoding='utf-8') as f:
        # Progress bar without a fixed total (since gzip line count is unknown)
        pbar = tqdm(desc="Ingesting records", unit=" rows")

        for line in f:
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue # Skip corrupted lines if any

            # Feature Selection: Only keep necessary columns to save RAM and Disk
            buffer.append({
                'user_id': record.get('user_id', ''),
                'book_id': record.get('book_id', ''),
                'rating': int(record.get('rating', 0)),
                'date_updated': record.get('date_updated', '')
            })

            if len(buffer) >= chunk_size:
                # Convert to PyArrow Table
                table = pa.Table.from_pylist(buffer, schema=schema)

                # Initialize writer on first chunk
                if writer is None:
                    writer = pq.ParquetWriter(output_path, schema)

                writer.write_table(table)

                # Strict Memory Cleanup
                buffer.clear()
                del table
                gc.collect()

                pbar.update(chunk_size)

        # Flush the remaining records in the buffer
        if buffer:
            table = pa.Table.from_pylist(buffer, schema=schema)
            if writer is None:
                writer = pq.ParquetWriter(output_path, schema)
            writer.write_table(table)
            pbar.update(len(buffer))

            buffer.clear()
            del table
            gc.collect()

        pbar.close()

    if writer:
        writer.close()

    log.info(f"Stage 1 completed. Parquet saved to: {output_path}")

# =============================================================================
# Execution and Verification Block
# =============================================================================
# if __name__ == "__main__":
#     try:
#         log.info("Testing Stage 1 Execution...")

#         # Execute the ingestion process
#         stage_1_ingestion(cfg)

#         # Verify output without loading the entire file into RAM
#         if cfg.ingested_parquet.exists():
#             pf = pq.ParquetFile(cfg.ingested_parquet)
#             total_rows = pf.metadata.num_rows
#             log.info("Verification Success!")
#             log.info(f"Total rows successfully written to Parquet: {total_rows:,}")
#         else:
#             log.error("Verification Failed! Output Parquet file not found.")

#     except Exception as e:
#         log.error(f"Error during Stage 1 execution: {str(e)}")

In [ ]:
# =============================================================================
# Cell 3: Stage 2 - Iterative K-Core Filtering (Bulletproof + Anti-Leak RAM)
# =============================================================================

import os
import gc
import logging
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq
import polars as pl

log = logging.getLogger("DataSplitPipeline")

def stage_2_kcore(cfg) -> None:
    """
    Bulletproof K-Core Filtering with Strict Memory Leak Prevention:
    - Uses DuckDB CHECKPOINT to flush Write-Ahead Logs (WAL).
    - Forces explicit Python Garbage Collection per iteration and per export batch.
    """
    if cfg.skip_stage2:
        log.info("Stage 2 skipped via configuration.")
        return

    log.info("Stage 2: Iterative K-Core Filtering (Anti-Leak Architecture)")

    input_file = str(cfg.ingested_parquet)
    output_file = str(cfg.kcore_parquet)
    db_file = str(cfg.data_dir / "kcore_workspace.db")

    if not os.path.exists(input_file):
        raise FileNotFoundError(f"Input file not found at: {input_file}")

    # Remove old workspace if it exists to ensure a fresh start
    if os.path.exists(db_file):
        try:
            os.remove(db_file)
            # Also remove WAL files if they were left behind
            if os.path.exists(db_file + ".wal"): os.remove(db_file + ".wal")
        except OSError:
            pass

    log.info("Initializing DuckDB database...")
    con = duckdb.connect(db_file)

    con.execute("PRAGMA threads=2")
    con.execute("PRAGMA memory_limit='8GB'")

    log.info("Loading Parquet data into DuckDB Engine...")
    con.execute(f"CREATE TABLE interactions AS SELECT * FROM read_parquet('{input_file}')")

    iteration = 1
    prev_user_count = -1
    prev_book_count = -1

    while True:
        log.info(f"--- K-Core Iteration {iteration} ---")

        log.info("Calculating retention thresholds...")
        con.execute(f"""
            CREATE TEMP TABLE valid_users AS
            SELECT user_id FROM interactions GROUP BY user_id HAVING COUNT(*) >= {cfg.k_user}
        """)
        con.execute(f"""
            CREATE TEMP TABLE valid_books AS
            SELECT book_id FROM interactions GROUP BY book_id HAVING COUNT(*) >= {cfg.k_book}
        """)

        current_user_count = con.execute("SELECT COUNT(*) FROM valid_users").fetchone()[0]
        current_book_count = con.execute("SELECT COUNT(*) FROM valid_books").fetchone()[0]

        log.info(f"Retained Users: {current_user_count:,} | Retained Books: {current_book_count:,}")

        if current_user_count == prev_user_count and current_book_count == prev_book_count:
            log.info("Convergence reached. Iteration completed.")
            con.execute("DROP TABLE valid_users")
            con.execute("DROP TABLE valid_books")
            break

        log.info("Filtering interactions table...")
        con.execute("""
            CREATE TABLE interactions_next AS
            SELECT i.* FROM interactions i
            INNER JOIN valid_users u ON i.user_id = u.user_id
            INNER JOIN valid_books b ON i.book_id = b.book_id
        """)

        con.execute("DROP TABLE interactions")
        con.execute("DROP TABLE valid_users")
        con.execute("DROP TABLE valid_books")
        con.execute("ALTER TABLE interactions_next RENAME TO interactions")

        # ANTI-LEAK MECHANISM 1: Force DuckDB to flush WAL to disk and free memory
        log.info("Checkpointing DB to free internal RAM...")
        con.execute("CHECKPOINT")

        # ANTI-LEAK MECHANISM 2: Force Python to clear any dangling query objects
        gc.collect()

        prev_user_count = current_user_count
        prev_book_count = current_book_count
        iteration += 1

    # =========================================================================
    # EXPORT PHASE (Memory Safe)
    # =========================================================================
    log.info("Exporting final table safely to Parquet...")
    query_result = con.execute("SELECT * FROM interactions")

    # 100K chunk size keeps RAM usage for PyArrow extremely flat
    record_batch_reader = query_result.fetch_record_batch(rows_per_batch=100_000)

    writer = None
    for batch in record_batch_reader:
        if writer is None:
            writer = pq.ParquetWriter(output_file, batch.schema, compression='snappy')
        writer.write_batch(batch)

        # ANTI-LEAK MECHANISM 3: Destroy the batch immediately, don't wait for loop to end
        del batch
        gc.collect()

    if writer:
        writer.close()

    con.close()

    # Final strict cleanup of the heavy DB files
    if os.path.exists(db_file): os.remove(db_file)
    if os.path.exists(db_file + ".wal"): os.remove(db_file + ".wal")

    log.info(f"Stage 2 completed perfectly. Final Parquet saved to: {output_file}")


# =============================================================================
# Execution and Verification Block
# =============================================================================
# if __name__ == "__main__":
#     try:
#         log.info("Testing Stage 2 Execution...")
#         stage_2_kcore(cfg)

#         if cfg.kcore_parquet.exists():
#             log.info("Running verification...")
#             total_rows = pl.scan_parquet(cfg.kcore_parquet).select(pl.len()).collect().item()
#             log.info("Verification Success! The Parquet file is structurally perfect.")
#             log.info(f"Total rows after K-Core filtering: {total_rows:,}")
#         else:
#             log.error("Verification Failed! Output Parquet file not found.")

#     except Exception as e:
#         log.error(f"Error during Stage 2 execution: {str(e)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
# =============================================================================
# Cell 4: Stage 3A & 3B - Chunk-based Temporal Split & Export
# =============================================================================

import os
import gc
import logging
import duckdb
import polars as pl
from tqdm.auto import tqdm

log = logging.getLogger("DataSplitPipeline")

def stage_3_chunked_split(cfg) -> None:
    """
    Chunk-based Processing Architecture:
    1. Calculates total interactions per user.
    2. Groups users into physical chunks (~1,000,000 interactions per chunk).
    3. Loops through each chunk: pulls data into RAM, sorts chronologically,
       calculates bounds, splits, and APPENDS directly to the final CSV files.
    4. Eliminates the need for massive out-of-core sorting.
    """
    if cfg.skip_stage3:
        log.info("Stage 3 skipped via configuration.")
        return

    log.info("Stage 3: Chunk-based Temporal Split & Export")

    input_file = str(cfg.kcore_parquet)
    if not os.path.exists(input_file):
        raise FileNotFoundError(f"Input file not found: {input_file}")

    # Xóa các file CSV cũ nếu có để tránh bị ghi nối thêm (append) sai lệch
    for file_path in cfg.outputs.values():
        if file_path.exists():
            file_path.unlink()

    log.info("Initializing DuckDB Engine...")
    con = duckdb.connect(database=':memory:')
    con.execute("PRAGMA threads=2")
    con.execute("PRAGMA memory_limit='6GB'")

    # -------------------------------------------------------------------------
    # BƯỚC 1: THỐNG KÊ VÀ PHÂN LÔ (CHUNKING)
    # -------------------------------------------------------------------------
    log.info("Step 1: Calculating user interaction counts & creating chunks...")

    # Lấy tổng số tương tác của từng user vào Polars DataFrame
    user_counts_df = con.execute(f"""
        SELECT user_id, COUNT(*) as total_rows
        FROM read_parquet('{input_file}')
        WHERE date_updated IS NOT NULL
        GROUP BY user_id
    """).pl()

    # Tính toán ID của từng Chunk (Mục tiêu ~1.000.000 dòng/chunk)
    CHUNK_SIZE = 5_000_000
    user_counts_df = (
        user_counts_df
        .with_columns(pl.col("total_rows").cum_sum().alias("cum_sum"))
        .with_columns((pl.col("cum_sum") // CHUNK_SIZE).alias("batch_id"))
    )

    num_batches = user_counts_df["batch_id"].max() + 1
    total_users = user_counts_df.height
    log.info(f"Divided {total_users:,} users into {num_batches} processing chunks.")

    # Đăng ký bảng mapping này vào DuckDB
    con.register("user_batch_mapping", user_counts_df.to_arrow())

    # Tạo một View ảo kết hợp dữ liệu gốc và batch_id (Không tốn RAM, không copy data)
    con.execute(f"""
        CREATE VIEW chunked_data AS
        SELECT t.user_id, t.book_id, CAST(t.rating AS INTEGER) AS rating, t.date_updated, m.batch_id, m.total_rows
        FROM read_parquet('{input_file}') t
        INNER JOIN user_batch_mapping m ON t.user_id = m.user_id
        WHERE t.date_updated IS NOT NULL
    """)

    # -------------------------------------------------------------------------
    # BƯỚC 2: XỬ LÝ TỪNG CHUNK VÀ GHI TRỰC TIẾP RA FILE (APPEND)
    # -------------------------------------------------------------------------
    log.info("Step 2: Processing and exporting chunks...")

    # Cấu hình danh sách các target export
    keep_cols = ["user_id", "book_id", "rating"]

    for batch_idx in tqdm(range(num_batches), desc="Processing Chunks"):
        # 2a. Kéo toàn bộ dữ liệu của nhóm user này vào RAM (Chỉ tốn ~20-50MB)
        df_chunk = con.execute(f"SELECT * FROM chunked_data WHERE batch_id = {batch_idx}").pl()

        if df_chunk.height == 0:
            continue

        # 2b. Sắp xếp siêu tốc trên RAM (Chronological sort)
        df_chunk = df_chunk.sort(["user_id", "date_updated"])

        # 2c. Tính toán các mốc chia bằng Window Function nội bộ trên RAM
        df_chunk = df_chunk.with_columns([
            pl.col("book_id").cum_count().over("user_id").alias("row_num"),
            (pl.col("total_rows") * cfg.train_ratio).floor().cast(pl.Int32).alias("train_end"),
            (pl.col("total_rows") * (cfg.train_ratio + cfg.valid_ratio)).floor().cast(pl.Int32).alias("valid_end")
        ])

        # 2d. Khởi tạo Boolean Masks (Điều kiện lọc)
        is_train = pl.col("row_num") <= pl.col("train_end")
        is_valid = (pl.col("row_num") > pl.col("train_end")) & (pl.col("row_num") <= pl.col("valid_end"))
        is_test = pl.col("row_num") > pl.col("valid_end")

        is_explicit = pl.col("rating").is_between(cfg.explicit_min_rating, cfg.explicit_max_rating)
        is_implicit = ~is_explicit
        is_pos = is_explicit & (pl.col("rating") >= cfg.positive_min_rating)
        is_neg = is_explicit & (pl.col("rating") <= cfg.negative_max_rating)

        # Định nghĩa các luồng ra file
        export_specs = [
            ("train_main", is_train, cfg.outputs["train_main"]),
            ("explicit_train", is_train & is_explicit, cfg.outputs["explicit_train"]),
            ("train_implicit", is_train & is_implicit, cfg.outputs["train_implicit"]),
            ("valid_pos", is_valid & is_pos, cfg.outputs["valid_pos"]),
            ("test_pos", is_test & is_pos, cfg.outputs["test_pos"]),
            ("test_neg", is_test & is_neg, cfg.outputs["test_neg"]),
            ("test_implicit", is_test & is_implicit, cfg.outputs["test_implicit"]),
            ("explicit_test", (is_valid | is_test) & is_explicit, cfg.outputs["explicit_test"]),
        ]

        # 2e. Ghi đè dòng (Append) thẳng vào các file CSV
        is_first_batch = (batch_idx == 0)

        for name, predicate, out_path in export_specs:
            sub_df = df_chunk.filter(predicate).select(keep_cols)

            if sub_df.height > 0:
                # Dùng mode="ab" (append binary) để ghi nối tiếp vào cuối file CSV
                with open(out_path, mode="ab" if not is_first_batch else "wb") as f:
                    sub_df.write_csv(f, include_header=is_first_batch)

        # 2f. Ép giải phóng bộ nhớ để đón Chunk tiếp theo
        del df_chunk
        gc.collect()

    con.close()
    log.info("Stage 3 completed perfectly! All 8 CSV files are generated.")

# =============================================================================
# Execution Block
# =============================================================================
if __name__ == "__main__":
    try:
        log.info("Testing Stage 3 Chunked Execution...")
        stage_3_chunked_split(cfg)

        # Verify một file đầu ra
        if cfg.outputs["train_main"].exists():
            log.info("Verification Success! 'train_main.csv' exists.")
            file_size_mb = cfg.outputs["train_main"].stat().st_size / (1024 * 1024)
            log.info(f"Size of train_main.csv: {file_size_mb:.2f} MB")
    except Exception as e:
        log.error(f"Error during Stage 3 execution: {str(e)}")

2026-04-18 11:05:43 [INFO] DataSplitPipeline: Testing Stage 3 Chunked Execution...
2026-04-18 11:05:43 [INFO] DataSplitPipeline: Stage 3: Chunk-based Temporal Split & Export
2026-04-18 11:05:43 [INFO] DataSplitPipeline: Initializing DuckDB Engine...
2026-04-18 11:05:43 [INFO] DataSplitPipeline: Step 1: Calculating user interaction counts & creating chunks...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04-18 11:06:04 [INFO] DataSplitPipeline: Divided 743,401 users into 45 processing chunks.
2026-04-18 11:06:05 [INFO] DataSplitPipeline: Step 2: Processing and exporting chunks...


Processing Chunks:   0%|          | 0/45 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04-18 13:14:14 [INFO] DataSplitPipeline: Stage 3 completed perfectly! All 8 CSV files are generated.
2026-04-18 13:14:14 [INFO] DataSplitPipeline: Verification Success! 'train_main.csv' exists.
2026-04-18 13:14:14 [INFO] DataSplitPipeline: Size of train_main.csv: 4398.88 MB


In [6]:
# =============================================================================
# Cell 5: Stage 4 - Data Quality Assurance (QA) & Validation
# =============================================================================

import logging
import polars as pl

log = logging.getLogger("DataSplitPipeline")

def stage_4_validation(cfg) -> None:
    """
    Performs critical Quality Assurance checks on the generated splits:
    1. Compares total interactions, users, and books against the original K-Core file.
    2. Prints the schema (columns) and row counts for every exported CSV.
    3. Checks for Data Leakage (Users in Test but not in Train).
    4. Checks for Sparse Users (Users in Train but not in Test).
    """
    log.info("Stage 4: Data Quality Assurance & Validation")

    # -------------------------------------------------------------------------
    # 1. THỐNG KÊ TỪ FILE GỐC (K-CORE PARQUET)
    # -------------------------------------------------------------------------
    log.info("Reading original K-Core statistics...")
    df_orig = pl.scan_parquet(cfg.kcore_parquet)

    orig_rows = df_orig.select(pl.len()).collect().item()
    orig_users = df_orig.select(pl.col("user_id").n_unique()).collect().item()
    orig_books = df_orig.select(pl.col("book_id").n_unique()).collect().item()

    print("\n" + "="*60)
    print("📊 BÁO CÁO KIỂM ĐỊNH DỮ LIỆU (DATA QA REPORT)")
    print("="*60)
    print(f"[FILE GỐC] K-Core Parquet:")
    print(f"   - Tổng tương tác : {orig_rows:,}")
    print(f"   - Tổng Users     : {orig_users:,}")
    print(f"   - Tổng Books     : {orig_books:,}")
    print("-" * 60)

    # -------------------------------------------------------------------------
    # 2. KIỂM TRA SCHEMA VÀ SỐ DÒNG CỦA CÁC FILE CSV ĐẦU RA
    # -------------------------------------------------------------------------
    print("[FILE ĐẦU RA] Chi tiết các tập dữ liệu đã chia:")

    # Lưu trữ tập hợp User để đối chiếu ở bước sau
    train_users_set = set()
    test_users_set = set()

    for name, path in cfg.outputs.items():
        if path.exists():
            # Quét file mà không nạp toàn bộ vào RAM
            df_lazy = pl.scan_csv(path)
            columns = df_lazy.columns
            row_count = df_lazy.select(pl.len()).collect().item()

            print(f"   ✓ {name}.csv")
            print(f"       + Số dòng : {row_count:,}")
            print(f"       + Trường  : {columns}")

            # Thu thập unique users cho tập Train Main
            if name == "train_main":
                train_users_set = set(df_lazy.select("user_id").unique().collect().get_column("user_id").to_list())

            # Thu thập unique users cho các tập Test/Valid (gom chung để kiểm tra leakage)
            if "test" in name or "valid" in name:
                users_in_file = set(df_lazy.select("user_id").unique().collect().get_column("user_id").to_list())
                test_users_set.update(users_in_file)
        else:
            print(f"   ❌ {name}.csv (KHÔNG TÌM THẤY)")

    print("-" * 60)

    # -------------------------------------------------------------------------
    # 3. KIỂM TRA ĐỐI CHIẾU USERS VÀ DATA LEAKAGE
    # -------------------------------------------------------------------------
    print("[ĐỐI CHIẾU USERS] Train vs Test/Valid:")

    users_in_train_not_test = train_users_set - test_users_set
    users_in_test_not_train = test_users_set - train_users_set

    print(f"   - Có trong Train, KHÔNG có trong Test : {len(users_in_train_not_test):,} users")
    print(f"   - Có trong Test, KHÔNG có trong Train : {len(users_in_test_not_train):,} users")

    # Đánh giá kết quả
    print("\n[ĐÁNH GIÁ KẾT QUẢ]")
    if len(users_in_test_not_train) == 0:
        print("   ✅ KHÔNG BỊ DATA LEAKAGE: 100% người dùng trong tập Test đều đã được mô hình học qua ở tập Train.")
    else:
        print("   ❌ CẢNH BÁO LỖI: Phát hiện người dùng có trong Test nhưng không có trong Train. Cần kiểm tra lại logic chia!")

    if len(users_in_train_not_test) > 0:
        print("   ℹ️ LƯU Ý BÌNH THƯỜNG: Việc có user nằm ở Train nhưng không có ở Test là ĐÚNG LOGIC.")
        print("      Lý do: Những user có tổng tương tác quá ít (VD: 3 tương tác), khi nhân với tỷ lệ split ")
        print("      sẽ không đủ làm tròn lên 1 dòng cho tập Test, do đó họ chỉ tồn tại ở tập Train.")

    print("="*60 + "\n")


# =============================================================================
# Execution Block
# =============================================================================
if __name__ == "__main__":
    try:
        stage_4_validation(cfg)
    except Exception as e:
        log.error(f"Error during Stage 4 validation: {str(e)}")

2026-04-18 13:57:21 [INFO] DataSplitPipeline: Stage 4: Data Quality Assurance & Validation
2026-04-18 13:57:21 [INFO] DataSplitPipeline: Reading original K-Core statistics...



📊 BÁO CÁO KIỂM ĐỊNH DỮ LIỆU (DATA QA REPORT)
[FILE GỐC] K-Core Parquet:
   - Tổng tương tác : 223,587,871
   - Tổng Users     : 743,401
   - Tổng Books     : 1,173,713
------------------------------------------------------------
[FILE ĐẦU RA] Chi tiết các tập dữ liệu đã chia:


/tmp/ipykernel_5753/589113447.py:52: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  columns = df_lazy.columns


   ✓ train_main.csv
       + Số dòng : 35,947,682
       + Trường  : ['user_id', 'book_id', 'rating']
   ✓ explicit_train.csv
       + Số dòng : 32,015,851
       + Trường  : ['user_id', 'book_id', 'rating']
   ✓ train_implicit.csv
       + Số dòng : 68,831,912
       + Trường  : ['user_id', 'book_id', 'rating']
   ✓ valid_pos.csv
       + Số dòng : 6,975,934
       + Trường  : ['user_id', 'book_id', 'rating']
   ✓ test_pos.csv
       + Số dòng : 6,872,706
       + Trường  : ['user_id', 'book_id', 'rating']
   ✓ test_neg.csv
       + Số dòng : 2,987,861
       + Trường  : ['user_id', 'book_id', 'rating']
   ✓ test_implicit.csv
       + Số dòng : 12,831,475
       + Trường  : ['user_id', 'book_id', 'rating']
   ✓ explicit_test.csv
       + Số dòng : 19,869,678
       + Trường  : ['user_id', 'book_id', 'rating']
------------------------------------------------------------
[ĐỐI CHIẾU USERS] Train vs Test/Valid:
   - Có trong Train, KHÔNG có trong Test : 1 users
   - Có trong Test, KHÔNG c

In [ ]:
# =============================================================================
# Cell 5: Stage 3B - Multi-Target CSV Export (Zero-RAM Streaming)
# =============================================================================

import os
import logging
import polars as pl

log = logging.getLogger("DataSplitPipeline")

def stage_3b_export_csvs(cfg) -> None:
    """
    Exports the ranked dataset into 8 separate CSV files based on temporal
    boundaries and feedback types. Uses Polars streaming (sink_csv) with
    Predicate Pushdown to ensure RAM usage remains near zero.
    """
    if cfg.skip_stage3:
        log.info("Stage 3 skipped via configuration.")
        return

    log.info("Stage 3B: Exporting Data to Final CSV Targets")
    ranked_file = str(cfg.data_dir / "interactions_ranked_tmp.parquet")

    if not os.path.exists(ranked_file):
        raise FileNotFoundError(f"Ranked file not found: {ranked_file}")

    # Create lazy execution graph
    lazy_df = pl.scan_parquet(ranked_file)

    # 1. Define Temporal Logical Conditions
    is_train = pl.col("row_num") <= pl.col("train_end")
    is_valid = (pl.col("row_num") > pl.col("train_end")) & (pl.col("row_num") <= pl.col("valid_end"))
    is_test = pl.col("row_num") > pl.col("valid_end")

    # 2. Define Feedback Logical Conditions
    is_explicit = pl.col("rating").is_between(cfg.explicit_min_rating, cfg.explicit_max_rating)
    is_implicit = pl.col("rating") == 0
    is_pos = is_explicit & (pl.col("rating") >= cfg.positive_min_rating)
    is_neg = is_explicit & (pl.col("rating") <= cfg.negative_max_rating)

    # 3. Map targets to their specific compound conditions
    export_specs = [
        ("train_main", is_train, cfg.outputs["train_main"]),
        ("explicit_train", is_train & is_explicit, cfg.outputs["explicit_train"]),
        ("train_implicit", is_train & is_implicit, cfg.outputs["train_implicit"]),
        ("valid_pos", is_valid & is_pos, cfg.outputs["valid_pos"]),
        ("test_pos", is_test & is_pos, cfg.outputs["test_pos"]),
        ("test_neg", is_test & is_neg, cfg.outputs["test_neg"]),
        ("test_implicit", is_test & is_implicit, cfg.outputs["test_implicit"]),
        ("explicit_test", (is_valid | is_test) & is_explicit, cfg.outputs["explicit_test"]),
    ]

    keep_cols = ["user_id", "book_id", "rating"]

    # 4. Stream data to CSV files
    for name, predicate, out_path in export_specs:
        log.info(f"Streaming {name} to CSV...")
        (
            lazy_df
            .filter(predicate)
            .select(keep_cols)
            .sink_csv(str(out_path))
        )

    log.info("All CSV exports completed successfully.")

    # 5. Cleanup the massive temporary ranked file to free disk space
    if os.path.exists(ranked_file):
        os.remove(ranked_file)
        log.info("Cleaned up temporary ranked parquet file.")


# =============================================================================
# Execution Block
# =============================================================================
if __name__ == "__main__":
    try:
        log.info("Testing Stage 3B Execution...")
        stage_3b_export_csvs(cfg)
    except Exception as e:
        log.error(f"Error during Stage 3B execution: {str(e)}")

In [ ]:
# import logging
# import duckdb
# import pyarrow.parquet as pq
# import gc
# from pathlib import Path

# # Cấu hình log cơ bản
# logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s]: %(message)s")
# log = logging.getLogger("DB_Export")

# def manual_export_from_db(db_path: str, output_parquet_path: str) -> None:
#     log.info(f"Opening existing DuckDB workspace: {db_path}")

#     # Kết nối vào file DB đã lưu
#     con = duckdb.connect(db_path)

#     # Kiểm tra xem bảng interactions có tồn tại không
#     tables = con.execute("SHOW TABLES").fetchall()
#     if ('interactions',) not in tables:
#         log.error("Table 'interactions' not found in the database!")
#         con.close()
#         return

#     total_rows = con.execute("SELECT COUNT(*) FROM interactions").fetchone()[0]
#     log.info(f"Found {total_rows:,} rows ready for export.")

#     log.info("Starting safe export to Parquet...")
#     query_result = con.execute("SELECT * FROM interactions")

#     # Rút từng batch 100.000 dòng
#     record_batch_reader = query_result.fetch_record_batch(rows_per_batch=100_000)

#     writer = None
#     for batch in record_batch_reader:
#         if writer is None:
#             writer = pq.ParquetWriter(output_parquet_path, batch.schema, compression='snappy')
#         writer.write_batch(batch)

#         del batch
#         gc.collect()

#     if writer:
#         writer.close()

#     con.close()
#     log.info(f"Manual export successfully completed! Saved to {output_parquet_path}")

# # Hướng dẫn sử dụng:
# # Thay thế đường dẫn cho đúng với môi trường của bạn
# manual_export_from_db(
#     db_path="/content/drive/My Drive/Thesis/book_recsys/data/kcore_workspace.db",
#     output_parquet_path="/content/drive/My Drive/Thesis/book_recsys/data/interactions_kcore.parquet"
# )

In [ ]:
# =============================================================================
# Cell: Verification (Zero-RAM Out-of-Core Check)
# =============================================================================

import os
import logging
import duckdb

logging.basicConfig(level=logging.INFO,force = True, format="%(asctime)s [%(levelname)s]: %(message)s")
log = logging.getLogger("DataVerification")

def verify_kcore_file_safe(parquet_path: str):
    """
    Verifies the integrity and logic of the K-Core filtered Parquet file
    using DuckDB's out-of-core aggregation to completely avoid RAM spikes.
    """
    if not os.path.exists(parquet_path):
        log.error(f"File not found: {parquet_path}")
        return

    log.info(f"Starting memory-safe verification for: {parquet_path}")

    # Khởi tạo DuckDB và giới hạn RAM cứng ở mức 8GB
    con = duckdb.connect(database=':memory:')
    con.execute("PRAGMA threads=2")
    con.execute("PRAGMA memory_limit='8GB'")

    try:
        # 1. Kiểm tra tổng số dòng
        log.info("Counting total rows...")
        total_rows = con.execute(f"SELECT COUNT(*) FROM '{parquet_path}'").fetchone()[0]

        # 2. Tìm số tương tác ít nhất của một User
        log.info("Calculating minimum interactions per user...")
        min_user_interactions = con.execute(f"""
            SELECT MIN(interact_count) FROM (
                SELECT COUNT(*) as interact_count
                FROM '{parquet_path}'
                WHERE rating > 3
                GROUP BY user_id
            )
        """).fetchone()[0]
        max_user_interaction = con.execute(f"""
          SELECT MAX(interact_count) FROM (
                SELECT COUNT(*) as interact_count
                FROM '{parquet_path}'
                GROUP BY user_id
            )
        """).fetchone()[0]

        # 3. Tìm số tương tác ít nhất của một Sách
        log.info("Calculating minimum interactions per book...")
        min_book_interactions = con.execute(f"""
            SELECT MIN(interact_count) FROM (
                SELECT COUNT(*) as interact_count
                FROM '{parquet_path}'
                GROUP BY book_id
            )
        """).fetchone()[0]

        # In Báo Cáo
        log.info("-" * 40)
        log.info("VERIFICATION REPORT")
        log.info("-" * 40)
        log.info(f"Total Rows: {total_rows:,}")
        log.info(f"Min interactions per User: {min_user_interactions}")
        log.info(f"Min interactions per Book: {min_book_interactions}")
        log.info(f"Max interactions per User: {max_user_interaction}")
        log.info("-" * 40)

        # Đánh giá logic
        if min_user_interactions >= cfg.k_user and min_book_interactions >= cfg.k_book:
            log.info("SUCCESS: K-Core constraints are perfectly met!")
        else:
            log.warning("WARNING: Constraints failed. Sparse data still exists.")

    except Exception as e:
        log.error(f"Verification failed: {e}")
    finally:
        con.close()

# Thực thi (Chỉ cần truyền đúng biến đường dẫn hoặc cfg.kcore_parquet)
if __name__ == "__main__":
    # Giả định biến cfg đã tồn tại từ Cell 1, nếu chạy riêng thì thay bằng string đường dẫn:
    # verify_kcore_file_safe("/content/drive/My Drive/Thesis/book_recsys/data/interactions_kcore.parquet")

    input_file = str(cfg.kcore_parquet)
    explicit_out = str(cfg.output_dir / "interactions_explicit.parquet")
    implicit_out = str(cfg.output_dir / "interactions_implicit.parquet")
    verify_kcore_file_safe(str(cfg.kcore_parquet))
    verify_kcore_file_safe(explicit_out)
    verify_kcore_file_safe(implicit_out)

2026-04-18 10:24:36,467 [INFO]: Starting memory-safe verification for: /content/drive/My Drive/Thesis/book_recsys/data/interactions_kcore.parquet
2026-04-18 10:24:36,683 [INFO]: Counting total rows...
2026-04-18 10:24:36,762 [INFO]: Calculating minimum interactions per user...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04-18 10:24:56,759 [INFO]: Calculating minimum interactions per book...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

2026-04-18 10:26:31,309 [INFO]: ----------------------------------------
2026-04-18 10:26:31,311 [INFO]: VERIFICATION REPORT
2026-04-18 10:26:31,313 [INFO]: ----------------------------------------
2026-04-18 10:26:31,314 [INFO]: Total Rows: 101,656,118
2026-04-18 10:26:31,315 [INFO]: Min interactions per User: 1
2026-04-18 10:26:31,317 [INFO]: Min interactions per Book: 1
2026-04-18 10:26:31,318 [INFO]: Max interactions per User: 36191
2026-04-18 10:26:31,319 [INFO]: ----------------------------------------
2026-04-18 10:26:31,321 [WARNING]: WARNING: Constraints failed. Sparse data still exists.
2026-04-18 10:26:31,349 [INFO]: Starting memory-safe verification for: /content/drive/My Drive/Thesis/book_recsys/data/processed_2/interactions_implicit.parquet
2026-04-18 10:26:31,367 [INFO]: Counting total rows...
2026-04-18 10:26:32,969 [INFO]: Calculating minimum interactions per user...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04-18 10:27:10,430 [INFO]: Calculating minimum interactions per book...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04-18 10:27:36,296 [INFO]: ----------------------------------------
2026-04-18 10:27:36,298 [INFO]: VERIFICATION REPORT
2026-04-18 10:27:36,298 [INFO]: ----------------------------------------
2026-04-18 10:27:36,299 [INFO]: Total Rows: 121,931,753
2026-04-18 10:27:36,300 [INFO]: Min interactions per User: None
2026-04-18 10:27:36,301 [INFO]: Min interactions per Book: 1
2026-04-18 10:27:36,302 [INFO]: Max interactions per User: 113199
2026-04-18 10:27:36,303 [INFO]: ----------------------------------------
2026-04-18 10:27:36,305 [ERROR]: Verification failed: '>=' not supported between instances of 'NoneType' and 'int'
